<a href="https://colab.research.google.com/github/dakshini01/ProdFusion/blob/main/codes/Data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
df = pd.read_csv("garments_worker_productivity.csv")

Saving garments_worker_productivity.csv to garments_worker_productivity (1).csv


In [10]:
# 2. Clean text columns first
df["department"] = df["department"].str.strip()
df["day"] = df["day"].str.strip()
df["quarter"] = df["quarter"].str.strip()

In [11]:
#Convert Date
df["date"] = pd.to_datetime(df["date"])




In [12]:
#Handle Missing WIP
df["wip_missing"] = df["wip"].isnull().astype(int)
df["wip"] = df["wip"].fillna(0)

In [13]:
# Feature Engineering: Interaction Terms
df["incentive_wip"] = df["incentive"] * df["wip"]
df["wip_workers"] = df["wip"] * df["no_of_workers"]
df["incentive_workers"] = df["incentive"] * df["no_of_workers"]

# Classification Label
df["low_productivity"] = (df["actual_productivity"] < 0.80).astype(int)

In [14]:
#drop unnecessary columns
df = df[[
    'date',
    'actual_productivity',
    'targeted_productivity',
    'smv',
    'wip',
    'over_time',
    'incentive',
    'idle_time',
    'idle_men',
    'no_of_style_change',
    'no_of_workers',
    'incentive_wip',
    'wip_workers',
    'incentive_workers',
    'low_productivity',
    'team',
    'department',
    'day',
    'wip_missing'
]]

In [15]:
#Sort by date only
df = df.sort_values(["team", "date"])
df.reset_index(drop=True, inplace=True)

In [16]:
#Encode Categorical Variables
df = pd.get_dummies(df, columns=['department', 'day'], drop_first=True)


In [17]:
#Convert bool columns to int BEFORE split
bool_cols = df.select_dtypes(include=['bool']).columns
df[bool_cols] = df[bool_cols].astype(int)

In [18]:
#Time based Train-Test Split
# Sort by date
df = df.sort_values(by="date")

split_date = df["date"].quantile(0.8)

train = df[df["date"] < split_date]
test  = df[df["date"] >= split_date]
# Features and target
X_train = train.drop(columns=["actual_productivity", "date"])
y_train = train["actual_productivity"]

X_test = test.drop(columns=["actual_productivity", "date"])
y_test = test["actual_productivity"]

# Check shapes
print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (943, 21)
Test: (254, 21)


In [19]:
#scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numeric_cols = [
    'targeted_productivity',
    'smv',
    'wip',
    'over_time',
    'incentive',
    'idle_time',
    'idle_men',
    'no_of_style_change',
    'no_of_workers',
    'incentive_wip',
    'wip_workers',
    'incentive_workers'
]

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [20]:
#Check Final Dataset Shape
print("Full dataset shape:", df.shape)
print("Training set shape:", train.shape)
print("Test set shape:", test.shape)

Full dataset shape: (1197, 23)
Training set shape: (943, 23)
Test set shape: (254, 23)


In [21]:
#Check Final Dataset Shape
print("Remaining missing values in full dataset:")
print(df.isnull().sum().sum())

print("Remaining missing values in training:")
print(X_train.isnull().sum().sum())

print("Remaining missing values in test:")
print(X_test.isnull().sum().sum())

Remaining missing values in full dataset:
0
Remaining missing values in training:
0
Remaining missing values in test:
0


In [24]:
df.head()



,date,actual_productivity,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,...,incentive_workers,low_productivity,team,wip_missing,department_sweing,day_Saturday,day_Sunday,day_Thursday,day_Tuesday,day_Wednesday
0,2015-01-01,0.886500,0.75,3.94,0.0,960,0,0.0,0,0,...,0.0,0,1,1,0,0,0,1,0,0
310,2015-01-01,0.521180,0.65,23.69,861.0,7200,0,0.0,0,0,...,0.0,1,4,0,1,0,0,1,0,0
309,2015-01-01,0.593056,0.75,3.94,0.0,2160,0,0.0,0,0,...,0.0,1,4,1,0,0,0,1,0,0
1010,2015-01-01,0.800570,0.80,11.41,968.0,3660,50,0.0,0,0,...,1525.0,0,11,0,1,0,0,1,0,0
1011,2015-01-01,0.436326,0.70,4.15,0.0,1440,0,0.0,0,0,...,0.0,1,11,1,0,0,0,1,0,0


In [26]:
#See Final Feature Columns
print("Final Feature Columns:")
print(X_train.columns)

Final Feature Columns:
Index(['targeted_productivity', 'smv', 'wip', 'over_time', 'incentive',
       'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers',
       'incentive_wip', 'wip_workers', 'incentive_workers', 'low_productivity',
       'team', 'wip_missing', 'department_sweing', 'day_Saturday',
       'day_Sunday', 'day_Thursday', 'day_Tuesday', 'day_Wednesday'],
      dtype='object')


In [27]:
#Confirm Time-Based Split Worked
print("Last training date:", train["date"].max())
print("First test date:", test["date"].min())

Last training date: 2015-02-25 00:00:00
First test date: 2015-02-26 00:00:00


In [28]:
#Get cleaned dataset
df.to_csv("cleaned_garment_data.csv", index=False)

from google.colab import files
files.download("cleaned_garment_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>